In [1]:
import transformers
print(transformers.__version__)
import torch
print(torch.__version__)
import evaluate
print(evaluate.__version__)


4.46.3
2.13.0+cpu
0.4.6


In [2]:
!pip install transformers==4.46.3
!pip install datasets
!pip install evaluate
!pip install jiwer
!pip install librosa
!pip install soundfile

In [3]:
import os
import glob
import time

import librosa
import torch
import evaluate

from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration
)

In [4]:
base_path = r"D:\AIML Assignments\Week-4\Question-6\free-spoken-digit-database"

labels = [
    "zero",
    "one",
    "two",
    "three",
    "four",
    "five",
    "six",
    "seven",
    "eight",
    "nine"
]

audio_files = []
ground_truth = []

for label in labels:
    files = glob.glob(os.path.join(base_path, label, "*.wav"))

    if len(files) > 0:
        audio_files.append(files[0])
        ground_truth.append(label)

print("Total files:", len(audio_files))

Total files: 10


In [5]:
for file, label in zip(audio_files, ground_truth):

    print(label)
    print(file)
    print()

zero
D:\AIML Assignments\Week-4\Question-6\free-spoken-digit-database\zero\004ae714_nohash_0.wav

one
D:\AIML Assignments\Week-4\Question-6\free-spoken-digit-database\one\004ae714_nohash_0.wav

two
D:\AIML Assignments\Week-4\Question-6\free-spoken-digit-database\two\00b01445_nohash_0.wav

three
D:\AIML Assignments\Week-4\Question-6\free-spoken-digit-database\three\00b01445_nohash_0.wav

four
D:\AIML Assignments\Week-4\Question-6\free-spoken-digit-database\four\004ae714_nohash_0.wav

five
D:\AIML Assignments\Week-4\Question-6\free-spoken-digit-database\five\004ae714_nohash_0.wav

six
D:\AIML Assignments\Week-4\Question-6\free-spoken-digit-database\six\00b01445_nohash_0.wav

seven
D:\AIML Assignments\Week-4\Question-6\free-spoken-digit-database\seven\004ae714_nohash_0.wav

eight
D:\AIML Assignments\Week-4\Question-6\free-spoken-digit-database\eight\012c8314_nohash_0.wav

nine
D:\AIML Assignments\Week-4\Question-6\free-spoken-digit-database\nine\00b01445_nohash_0.wav



In [6]:
audio_file = audio_files[0]

audio_array, sampling_rate = librosa.load(
    audio_file,
    sr=16000
)

print("Sampling Rate :", sampling_rate)
print("Duration :", len(audio_array)/sampling_rate)

Sampling Rate : 16000
Duration : 0.7895


In [7]:
processor = WhisperProcessor.from_pretrained(
    "openai/whisper-tiny"
)

model = WhisperForConditionalGeneration.from_pretrained(
    "openai/whisper-tiny"
)

model.eval()

print("Model Loaded Successfully")

Model Loaded Successfully


In [8]:
inputs = processor(
    audio_array,
    sampling_rate=16000,
    return_tensors="pt"
)

In [9]:
start = time.time()

with torch.no_grad():

    predicted_ids = model.generate(
        inputs["input_features"]
    )

full_time = time.time() - start

full_text = processor.batch_decode(
    predicted_ids,
    skip_special_tokens=True
)[0]

print(full_text)

Due to a bug fix in https://github.com/huggingface/transformers/pull/28687 transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English.This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`.
Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.43.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


 Zebra.


In [10]:
quantized_model = torch.quantization.quantize_dynamic(
    model,
    {torch.nn.Linear},
    dtype=torch.qint8
)

print("Quantized Successfully")

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_13772\1744342749.py:1: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  quantized_model = torch.quantization.quantize_dynamic(
D:\Anaconda\envs\opencv-env\Lib\site-packages\torch\ao\nn\quantized\modules\utils.py:72: UserWarning: torch.quantize_per_tensor, torch.quantize_per_channel and other quantized tensor creation functions tha

Quantized Successfully


In [11]:
start = time.time()

with torch.no_grad():

    predicted_ids = quantized_model.generate(
        inputs["input_features"]
    )

quant_time = time.time() - start

quant_text = processor.batch_decode(
    predicted_ids,
    skip_special_tokens=True
)[0]

print(quant_text)

 zebra.


In [12]:
wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")

wer_full = wer_metric.compute(
    predictions=[full_text],
    references=[ground_truth[0]]
)

cer_full = cer_metric.compute(
    predictions=[full_text],
    references=[ground_truth[0]]
)

wer_quant = wer_metric.compute(
    predictions=[quant_text],
    references=[ground_truth[0]]
)

cer_quant = cer_metric.compute(
    predictions=[quant_text],
    references=[ground_truth[0]]
)

In [13]:
torch.save(model.state_dict(),"full_model.pth")
torch.save(quantized_model.state_dict(),"quantized_model.pth")

full_size = os.path.getsize("full_model.pth")/(1024*1024)
quant_size = os.path.getsize("quantized_model.pth")/(1024*1024)

In [14]:
print("="*50)

print("Ground Truth :", ground_truth[0])

print()

print("Full Model Prediction :", full_text)
print("Quantized Prediction :", quant_text)

print()

print("WER Full :", wer_full)
print("CER Full :", cer_full)

print()

print("WER Quantized :", wer_quant)
print("CER Quantized :", cer_quant)

print()

print("Full Inference Time :", round(full_time,4),"seconds")
print("Quantized Inference Time :", round(quant_time,4),"seconds")

print()

print("Full Model Size :", round(full_size,2),"MB")
print("Quantized Model Size :", round(quant_size,2),"MB")

print("="*50)

Ground Truth : zero

Full Model Prediction :  Zebra.
Quantized Prediction :  zebra.

WER Full : 1.0
CER Full : 1.0

WER Quantized : 1.0
CER Quantized : 0.75

Full Inference Time : 2.6089 seconds
Quantized Inference Time : 1.4455 seconds

Full Model Size : 144.1 MB
Quantized Model Size : 115.9 MB


Model: Whisper Tiny (OpenAI) from Hugging Face. It is a lightweight automatic speech recognition model capable of converting speech into text. It supports multiple languages and is suitable for fast inference on CPU systems.

Inference Pipeline

Load audio files.

Resample audio to 16 kHz using Librosa.

Extract features using Whisper Processor.

Generate transcription using the full Whisper model.

Quantize the model using PyTorch Dynamic Quantization.

Generate transcription using the quantized model.

Compare WER, CER, inference time, and model size.
